# 🛣️ Machine Learning for Pavement Deterioration & Roughness (IRI) Prediction
### Case Study: National Highway N-5 (Sangjani Toll Plaza to Attock Bridge)
**Author:** [Muhammad Umer Basit](https://github.com/MuhammadUmerBasit)

---
## 1. Project Background & Motivation
Road pavement deterioration is driven by a complex interaction between cumulative traffic loading (AADT / ESALs) and structural pavement capacity (Effective Structural Number, $SN_{eff}$). 
The **International Roughness Index (IRI)** is the global standard metric used by the World Bank and highway authorities to assess ride quality, operating costs, and structural rehabilitation timing.

In this notebook, we analyze real field data collected by the **National Highway Authority (NHA)** using:
- **Heavy Weight Deflectometer (HWD)** deflection data backcalculated to Total $SN$
- **Classified Traffic Volume Counts** (AADT)
- **Laser Profilometer Surface Roughness** (IRI in in/mi and m/km)


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

sns.set_theme(style="whitegrid")
print("Libraries imported successfully.")

## 2. Load & Explore the Highway Dataset

In [ ]:
csv_path = "../dataset/pavement_traffic_dataset.csv"
df = pd.read_csv(csv_path)
print(f"Total Highway Segments Loaded: {len(df)}")
df.head(10)

In [ ]:
df.describe().round(2)

## 3. Exploratory Data Analysis & Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution of IRI
sns.histplot(df["IRI_in_per_mi"], kde=True, color="#3498db", ax=axes[0])
axes[0].set_title("Distribution of Pavement Roughness (IRI in/mi)")
axes[0].axvline(170, color="red", linestyle="--", label="FHWA Poor Threshold (>170)")
axes[0].legend()

# Boxplot by Section
sns.boxplot(data=df, x="Highway_Section", y="IRI_in_per_mi", palette="Set2", ax=axes[1])
axes[1].set_title("Roughness Variation Across N-5 Highway Sections")
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=25, ha="right")

plt.tight_layout()
plt.show()

## 4. Machine Learning Model Training & Evaluation

In [ ]:
X = df[["Structural_Number_SN", "AADT"]].values
y = df["IRI_in_per_mi"].values

models = {
    "Multi-Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(max_depth=4, random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=100, max_depth=3, random_state=42)
}

print(f"{'Model':<25} | {'MAE (in/mi)':<12} | {'RMSE':<10} | {'R2 Score':<8}")
print("-" * 60)
for name, model in models.items():
    model.fit(X, y)
    y_pred = model.predict(X)
    mae = mean_absolute_error(y, y_pred)
    rmse = np.sqrt(mean_squared_error(y, y_pred))
    r2 = r2_score(y, y_pred)
    print(f"{name:<25} | {mae:<12.2f} | {rmse:<10.2f} | {r2:<8.4f}")

## 5. Domain Piecewise Model Formulation

In [ ]:
def predict_piecewise_domain(sn, aadt):
    if aadt < 400:
        return sn * 0.25233439 + aadt * 0.00408931 + 175.062028028023
    elif aadt < 1500:
        return 3.24444 * sn + 144.631
    elif aadt < 2500:
        return -0.1235 * sn + 202.94
    elif aadt < 3500:
        return -5.886 * sn + 220.602
    elif aadt < 5000:
        return 13.75103 * sn + 104.9813
    elif aadt < 8000:
        return 0.25946 * sn + 177.913
    else:
        return sn * 0.25233439 + aadt * 0.00408931 + 175.062028028023

# Evaluate domain model
y_pred_domain = np.array([predict_piecewise_domain(sn, aadt) for sn, aadt in X])
print(f"Domain Model MAE:  {mean_absolute_error(y, y_pred_domain):.2f} in/mi")
print(f"Domain Model RMSE: {np.sqrt(mean_squared_error(y, y_pred_domain)):.2f} in/mi")
print(f"Domain Model R2:   {r2_score(y, y_pred_domain):.4f}")

## 6. Interactive Prediction & Maintenance Recommendation

In [ ]:
sample_sn = 8.5
sample_aadt = 4500
pred_iri = predict_piecewise_domain(sample_sn, sample_aadt)

print(f"Input SN: {sample_sn}, Input AADT: {sample_aadt} veh/day")
print(f"Predicted IRI: {pred_iri:.2f} in/mi ({pred_iri * 0.0157828:.3f} m/km)")
if pred_iri < 95:
    print("Condition: Good -> Action: Routine Maintenance")
elif pred_iri <= 170:
    print("Condition: Fair -> Action: Preventative Resurfacing")
else:
    print("Condition: Poor -> Action: Structural Rehabilitation / Reconstruction")